# Lab 1：PyTorch 模型训练与推理

## 实验目标

- 理解图像分类模型的训练、评估和推理过程
- 掌握卷积神经网络的基本结构，以及 logits、概率和类别预测之间的关系
- 能够在 Ascend NPU 上完成反向传播和参数更新
- 能够保存并加载 PyTorch checkpoint，验证恢复后的模型并完成分类推理

## 实验环境

| 项目 | 配置 |
| --- | --- |
| NPU | 单卡 Ascend 910B3（Atlas A2） |
| CANN | 9.0.0 |
| Python | 3.11.4 |
| 关键工具 | PyTorch、torch_npu、Jupyter Notebook、Matplotlib |

## 实验原理

本实验把 16×16 的灰度图分为竖线、横线和斜线三类。输入张量的形状是 `[N, C, H, W]`，分别表示批量大小、通道数、图像高度和图像宽度。

`ShapeCNN` 使用两组卷积、激活和池化操作提取图像特征，再由全连接层输出三个 logits。训练时，`CrossEntropyLoss` 直接根据 logits 和类别标签计算损失；推理时，再用 Softmax 把 logits 转为类别概率。

反向传播将梯度写入各参数的 `.grad`，优化器随后根据梯度更新参数值。训练结束后，模型的 `state_dict` 和必要的训练信息会写入 checkpoint。加载 checkpoint 时先创建结构相同的新模型，再恢复参数并切换到评估模式。

## 实验流程

### 1. 初始化环境

设置随机种子和 NPU 设备，并检查当前 Kernel 能否访问 Ascend NPU。

In [ ]:
import os
import platform
import sys

os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "0"
cann_home = os.environ.get("ASCEND_HOME_PATH") or os.environ.get("ASCEND_TOOLKIT_HOME")
if cann_home and "ASCEND_OPP_PATH" not in os.environ:
    os.environ["ASCEND_OPP_PATH"] = cann_home.rstrip("/") + "/opp"

import torch
import torch_npu

SEED = 42
DEVICE = torch.device("npu:0")

if not torch.npu.is_available() or torch.npu.device_count() < 1:
    raise RuntimeError("当前 Kernel 未检测到可用的 Ascend NPU。")

torch.npu.set_device(DEVICE)
torch.manual_seed(SEED)
torch.npu.manual_seed_all(SEED)

print(f"[ENV] python={platform.python_version()} torch={torch.__version__}")
print(f"[ENV] kernel={sys.executable}")
print(f"[ENV] ASCEND_OPP_PATH={os.environ.get('ASCEND_OPP_PATH', '<未设置>')}")
print(f"[ENV] device={DEVICE} name={torch.npu.get_device_name(0)}")


### 2. 准备数据

`make_shape_dataset` 生成带随机位移和噪声的三类图像。每张图像的形状是 `[1, 16, 16]`，标签类型是 `int64`。`TensorDataset` 负责配对图像和标签，`DataLoader` 将样本整理为 batch。

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

IMAGE_SIZE = 16
CLASS_NAMES = ["竖线", "横线", "斜线"]
SAMPLES_PER_CLASS = 240


def make_shape_dataset(seed=SEED, samples_per_class=SAMPLES_PER_CLASS):
    generator = torch.Generator().manual_seed(seed)
    images, labels = [], []

    for class_id in range(len(CLASS_NAMES)):
        for _ in range(samples_per_class):
            image = torch.zeros(IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
            shift = int(torch.randint(-2, 3, (1,), generator=generator).item())

            if class_id == 0:
                column = IMAGE_SIZE // 2 + shift
                image[2:14, column - 1:column + 1] = 1.0
            elif class_id == 1:
                row = IMAGE_SIZE // 2 + shift
                image[row - 1:row + 1, 2:14] = 1.0
            else:
                for row in range(2, 14):
                    column = row + shift
                    if 1 <= column < IMAGE_SIZE - 1:
                        image[row, column:column + 2] = 1.0

            noise = torch.randn(image.shape, generator=generator) * 0.12
            images.append((image + noise).clamp(0.0, 1.0).unsqueeze(0))
            labels.append(class_id)

    images = torch.stack(images)
    labels = torch.tensor(labels, dtype=torch.long)
    order = torch.randperm(len(labels), generator=generator)
    images, labels = images[order], labels[order]
    split = int(len(labels) * 0.8)
    return images[:split], labels[:split], images[split:], labels[split:]


def make_loader(images, labels, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TensorDataset(images, labels),
        batch_size=32,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
    )


x_train, y_train, x_test, y_test = make_shape_dataset()
train_loader = make_loader(x_train, y_train, shuffle=True)
test_loader = make_loader(x_test, y_test, shuffle=False)
first_images, first_labels = next(iter(train_loader))

print(f"[DATA] train={len(x_train)} test={len(x_test)} classes={CLASS_NAMES}")
print("首个 batch images:", first_images.shape, first_images.dtype)
print("首个 batch labels:", first_labels.shape, first_labels.dtype)

assert tuple(first_images.shape) == (32, 1, IMAGE_SIZE, IMAGE_SIZE)
assert first_images.dtype == torch.float32
assert first_labels.dtype == torch.int64


### 3. 定义模型

图像依次经过两组卷积和池化操作，特征尺寸由 `[N, 1, 16, 16]` 变为 `[N, 16, 4, 4]`。展平后的特征进入全连接层，最终得到三个 logits。

In [ ]:
from torch import nn


class ShapeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 32),
            nn.ReLU(),
            nn.Linear(32, len(CLASS_NAMES)),
        )

    def forward(self, images):
        features = self.features(images)
        return self.classifier(features)


model = ShapeCNN().to(DEVICE)
sample_images = first_images.to(DEVICE)
sample_logits = model(sample_images)

print("输入:", sample_images.shape, sample_images.device)
print("输出 logits:", sample_logits.shape, sample_logits.device)
print("预测类别:", sample_logits.argmax(dim=1)[:8].cpu().tolist())
print("可训练参数量:", sum(parameter.numel() for parameter in model.parameters()))

assert tuple(sample_logits.shape) == (32, len(CLASS_NAMES))


### 4. 检查反向传播

先用一个 batch 走完清除梯度、前向计算、损失计算、反向传播和参数更新。代码会比较更新前后的参数，确认优化器已经生效。

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

probe_images = first_images.to(DEVICE)
probe_labels = first_labels.to(DEVICE)
first_parameter = next(model.parameters())
parameter_before = first_parameter.detach().clone()

optimizer.zero_grad(set_to_none=True)
probe_logits = model(probe_images)
probe_loss = loss_fn(probe_logits, probe_labels)
probe_loss.backward()

print("probe logits:", probe_logits.shape)
print("probe loss:", float(probe_loss.item()))
print("第一个参数:", first_parameter.shape)
print("对应梯度:", first_parameter.grad.shape)
print("梯度有限:", bool(torch.isfinite(first_parameter.grad).all().item()))

optimizer.step()
parameter_change = float((first_parameter.detach() - parameter_before).abs().max().item())
print(f"参数最大变化: {parameter_change:.8f}")
assert parameter_change > 0


### 5. 训练并评估模型

`train_epoch` 在训练模式下更新参数，`evaluate` 在评估模式下统计损失和准确率。

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)


def train_epoch(model, data_loader):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in data_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.shape[0]
        loss_sum += float(loss.item()) * batch_size
        correct += int((logits.argmax(dim=1) == labels).sum().item())
        total += batch_size
    return loss_sum / total, correct / total


def evaluate(model, data_loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)
            batch_size = labels.shape[0]
            loss_sum += float(loss.item()) * batch_size
            correct += int((logits.argmax(dim=1) == labels).sum().item())
            total += batch_size
    return loss_sum / total, correct / total


运行 8 个 epoch，并在每轮训练后评估模型。

In [ ]:
EPOCHS = 8
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = train_epoch(model, train_loader)
    test_loss, test_accuracy = evaluate(model, test_loader)
    history.append((train_loss, train_accuracy, test_loss, test_accuracy))
    print(
        f"[EPOCH {epoch:02d}/{EPOCHS:02d}] "
        f"train_loss={train_loss:.4f} train_acc={train_accuracy:.4f} "
        f"test_loss={test_loss:.4f} test_acc={test_accuracy:.4f}"
    )

torch.npu.synchronize()


### 6. 保存 checkpoint

保存模型参数和训练信息。参数张量先移到 CPU，读取 checkpoint 时不必依赖训练设备。

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("outputs/lab01")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "shape_cnn.pt"

final_test_loss, final_test_accuracy = evaluate(model, test_loader)
cpu_state_dict = {
    name: tensor.detach().cpu()
    for name, tensor in model.state_dict().items()
}

torch.save({
    "model_state_dict": cpu_state_dict,
    "model_name": "ShapeCNN",
    "image_size": IMAGE_SIZE,
    "class_names": CLASS_NAMES,
    "samples_per_class": SAMPLES_PER_CLASS,
    "seed": SEED,
    "epochs": EPOCHS,
    "torch_version": str(torch.__version__),
    "trained_device": str(DEVICE),
    "final_test_loss": final_test_loss,
    "final_test_accuracy": final_test_accuracy,
}, CHECKPOINT_PATH)

checkpoint_ok = CHECKPOINT_PATH.is_file() and CHECKPOINT_PATH.stat().st_size > 0
loss_decreased = history[-1][0] < history[0][0]
accuracy_ok = final_test_accuracy >= 0.95

print(f"[CHECK] final_test_loss={final_test_loss:.4f} final_test_accuracy={final_test_accuracy:.4f}")
print(f"[ARTIFACT] checkpoint={CHECKPOINT_PATH} size={CHECKPOINT_PATH.stat().st_size}")

if loss_decreased and accuracy_ok and checkpoint_ok:
    print("[PASS] CNN 分类训练完成，checkpoint 已保存。")
else:
    print("[FAIL]", {
        "loss_decreased": loss_decreased,
        "accuracy_ok": accuracy_ok,
        "checkpoint_ok": checkpoint_ok,
    })


### 7. 重新创建数据与模型

重新初始化环境，再生成数据并定义 `ShapeCNN`。这一步不沿用训练阶段内存中的模型对象。数据函数使用相同的随机种子，因此得到的测试数据与保存 checkpoint 时使用的测试划分一致。

In [ ]:
import os
import platform
import sys

os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "0"
cann_home = os.environ.get("ASCEND_HOME_PATH") or os.environ.get("ASCEND_TOOLKIT_HOME")
if cann_home and "ASCEND_OPP_PATH" not in os.environ:
    os.environ["ASCEND_OPP_PATH"] = cann_home.rstrip("/") + "/opp"

import torch
import torch_npu

SEED = 42
DEVICE = torch.device("npu:0")

if not torch.npu.is_available() or torch.npu.device_count() < 1:
    raise RuntimeError("当前 Kernel 未检测到可用的 Ascend NPU。")

torch.npu.set_device(DEVICE)
torch.manual_seed(SEED)
torch.npu.manual_seed_all(SEED)

print(f"[ENV] python={platform.python_version()} torch={torch.__version__}")
print(f"[ENV] kernel={sys.executable}")
print(f"[ENV] ASCEND_OPP_PATH={os.environ.get('ASCEND_OPP_PATH', '<未设置>')}")
print(f"[ENV] device={DEVICE} name={torch.npu.get_device_name(0)}")


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

IMAGE_SIZE = 16
CLASS_NAMES = ["竖线", "横线", "斜线"]
SAMPLES_PER_CLASS = 240


def make_shape_dataset(seed=SEED, samples_per_class=SAMPLES_PER_CLASS):
    generator = torch.Generator().manual_seed(seed)
    images, labels = [], []

    for class_id in range(len(CLASS_NAMES)):
        for _ in range(samples_per_class):
            image = torch.zeros(IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
            shift = int(torch.randint(-2, 3, (1,), generator=generator).item())

            if class_id == 0:
                column = IMAGE_SIZE // 2 + shift
                image[2:14, column - 1:column + 1] = 1.0
            elif class_id == 1:
                row = IMAGE_SIZE // 2 + shift
                image[row - 1:row + 1, 2:14] = 1.0
            else:
                for row in range(2, 14):
                    column = row + shift
                    if 1 <= column < IMAGE_SIZE - 1:
                        image[row, column:column + 2] = 1.0

            noise = torch.randn(image.shape, generator=generator) * 0.12
            images.append((image + noise).clamp(0.0, 1.0).unsqueeze(0))
            labels.append(class_id)

    images = torch.stack(images)
    labels = torch.tensor(labels, dtype=torch.long)
    order = torch.randperm(len(labels), generator=generator)
    images, labels = images[order], labels[order]
    split = int(len(labels) * 0.8)
    return images[:split], labels[:split], images[split:], labels[split:]


def make_loader(images, labels, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TensorDataset(images, labels),
        batch_size=32,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
    )


x_train, y_train, x_test, y_test = make_shape_dataset()
train_loader = make_loader(x_train, y_train, shuffle=True)
test_loader = make_loader(x_test, y_test, shuffle=False)
first_images, first_labels = next(iter(train_loader))

print(f"[DATA] train={len(x_train)} test={len(x_test)} classes={CLASS_NAMES}")
print("首个 batch images:", first_images.shape, first_images.dtype)
print("首个 batch labels:", first_labels.shape, first_labels.dtype)

assert tuple(first_images.shape) == (32, 1, IMAGE_SIZE, IMAGE_SIZE)
assert first_images.dtype == torch.float32
assert first_labels.dtype == torch.int64


In [ ]:
from pathlib import Path
from torch import nn


class ShapeCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 32),
            nn.ReLU(),
            nn.Linear(32, len(CLASS_NAMES)),
        )

    def forward(self, images):
        return self.classifier(self.features(images))


### 8. 加载 checkpoint

创建新的 `ShapeCNN`，加载保存的参数，然后将模型移到 NPU 并切换到评估模式。

In [ ]:
CHECKPOINT_PATH = Path("outputs/lab01/shape_cnn.pt")
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"未找到 {CHECKPOINT_PATH}，请先运行前面的训练与保存步骤。")

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
restored_model = ShapeCNN()
restored_model.load_state_dict(checkpoint["model_state_dict"])
restored_model = restored_model.to(DEVICE)
restored_model.eval()

print(f"[LOAD] checkpoint={CHECKPOINT_PATH} size={CHECKPOINT_PATH.stat().st_size}")
print(f"[LOAD] model={checkpoint['model_name']} classes={checkpoint['class_names']}")
print(f"[LOAD] trained_device={checkpoint['trained_device']} current_device={DEVICE}")
print(f"[LOAD] saved_accuracy={checkpoint['final_test_accuracy']:.4f}")


### 9. 评估恢复后的模型

在相同的测试划分上评估恢复后的模型，并比较本次准确率与 checkpoint 中记录的准确率。两者应当一致。

In [ ]:
loss_fn = nn.CrossEntropyLoss()


def evaluate(model, data_loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits = model(images)
            loss = loss_fn(logits, labels)
            batch_size = labels.shape[0]
            loss_sum += float(loss.item()) * batch_size
            correct += int((logits.argmax(dim=1) == labels).sum().item())
            total += batch_size
    return loss_sum / total, correct / total


reload_loss, reload_accuracy = evaluate(restored_model, test_loader)
accuracy_gap = abs(reload_accuracy - float(checkpoint["final_test_accuracy"]))

print(f"[EVAL] test_loss={reload_loss:.4f} test_accuracy={reload_accuracy:.4f}")
print(f"[EVAL] accuracy_gap_vs_saved={accuracy_gap:.8f}")


### 10. 执行分类推理

读取一个测试 batch，将 logits 转为概率，并输出前 10 个样本的真实类别、预测类别和置信度。

In [ ]:
inference_images, inference_labels = next(iter(test_loader))

with torch.no_grad():
    inference_logits = restored_model(inference_images.to(DEVICE))
    inference_probabilities = torch.softmax(inference_logits, dim=1).cpu()
    inference_predictions = inference_probabilities.argmax(dim=1)

print("序号  真实类别  预测类别  置信度")
for index in range(10):
    true_id = int(inference_labels[index])
    predicted_id = int(inference_predictions[index])
    confidence = float(inference_probabilities[index, predicted_id])
    print(
        f"{index:>4}  {CLASS_NAMES[true_id]:>8}  "
        f"{CLASS_NAMES[predicted_id]:>8}  {confidence:.4f}"
    )

batch_correct = int((inference_predictions == inference_labels).sum())
print(f"[INFER] batch_shape={tuple(inference_images.shape)} device={DEVICE}")
print(f"[INFER] batch_correct={batch_correct}/{len(inference_labels)}")


### 11. 可视化推理结果

绘制 12 张测试图像，标注真实类别、预测类别和置信度，并将结果保存为 `outputs/lab01/inference_examples.png`。

In [ ]:
import matplotlib.pyplot as plt

CLASS_NAMES_EN = ["Vertical", "Horizontal", "Diagonal"]
IMAGE_GRID_PATH = Path("outputs/lab01/inference_examples.png")

figure, axes = plt.subplots(3, 4, figsize=(8, 6))
for index, axis in enumerate(axes.flat):
    true_id = int(inference_labels[index])
    predicted_id = int(inference_predictions[index])
    confidence = float(inference_probabilities[index, predicted_id])

    axis.imshow(inference_images[index, 0].numpy(), cmap="gray", vmin=0.0, vmax=1.0)
    axis.set_title(
        f"True: {CLASS_NAMES_EN[true_id]}\n"
        f"Pred: {CLASS_NAMES_EN[predicted_id]} ({confidence:.2f})",
        fontsize=9,
    )
    axis.axis("off")

figure.suptitle("ShapeCNN test predictions", fontsize=13)
figure.tight_layout()
figure.savefig(IMAGE_GRID_PATH, dpi=150, bbox_inches="tight")
plt.show()

image_grid_ok = IMAGE_GRID_PATH.is_file() and IMAGE_GRID_PATH.stat().st_size > 0
print(f"[ARTIFACT] test_images={IMAGE_GRID_PATH} size={IMAGE_GRID_PATH.stat().st_size}")


## 实验总结

本实验完成了 `ShapeCNN` 在 Ascend NPU 上的训练、评估、保存和恢复。训练阶段通过反向传播更新模型参数；推理阶段加载 checkpoint，将 logits 转为概率，并输出分类结果和可视化图像。